# ANN Lab Task 02 - Logistic Regression
## Dataset 1: Breast Cancer | Dataset 2: Wine Quality
### Gradio UI + Flask UI

## Step 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print('Libraries imported successfully')

---
## Dataset 1 - Breast Cancer
**Target:** 0 = Malignant, 1 = Benign

### Step 2 - Load Dataset 1

In [ ]:
df1 = pd.read_csv(r"C:\Users\Geo Computer\OneDrive\Desktop\ANN Lab Task 02\breast_cancer.csv")

print('Shape:', df1.shape)
print(df1['target'].value_counts())
df1.head()

### Step 3 - Visualize Dataset 1

In [ ]:
plt.figure(figsize=(5, 4))
df1['target'].value_counts().plot(kind='bar', color=['red', 'green'], edgecolor='black')
plt.title('Breast Cancer - Class Count')
plt.xlabel('0 = Malignant, 1 = Benign')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='target', y='mean radius', data=df1, palette=['red', 'green'])
plt.title('Mean Radius by Class')
plt.xlabel('0 = Malignant, 1 = Benign')
plt.show()

### Step 4 - Preprocess & Train Model 1

In [ ]:
X1 = df1.drop('target', axis=1)
y1 = df1['target']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

scaler1 = StandardScaler()
X1_train = scaler1.fit_transform(X1_train)
X1_test  = scaler1.transform(X1_test)

model1 = LogisticRegression(max_iter=10000)
model1.fit(X1_train, y1_train)

print('Model 1 trained!')

### Step 5 - Evaluate Model 1

In [ ]:
y1_pred = model1.predict(X1_test)
acc1 = accuracy_score(y1_test, y1_pred)

print('Accuracy:', round(acc1 * 100, 2), '%')
print()
print(classification_report(y1_test, y1_pred, target_names=['Malignant', 'Benign']))

In [ ]:
cm1 = confusion_matrix(y1_test, y1_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
plt.title('Confusion Matrix - Breast Cancer')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

---
## Dataset 2 - Wine Quality
**Target:** 0 = Low Quality (score < 7), 1 = High Quality (score >= 7)

### Step 6 - Load Dataset 2

In [ ]:
df2 = pd.read_csv(r"C:\Users\Geo Computer\OneDrive\Desktop\ANN Lab Task 02\wine_quality.csv")

# Make it binary: quality >= 7 is High (1), else Low (0)
df2['target'] = (df2['quality'] >= 7).astype(int)
df2 = df2.drop('quality', axis=1)

print('Shape:', df2.shape)
print(df2['target'].value_counts())
df2.head()

### Step 7 - Visualize Dataset 2

In [ ]:
plt.figure(figsize=(5, 4))
df2['target'].value_counts().plot(kind='bar', color=['orange', 'purple'], edgecolor='black')
plt.title('Wine Quality - Class Count')
plt.xlabel('0 = Low Quality, 1 = High Quality')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='target', y='alcohol', data=df2, palette=['orange', 'purple'])
plt.title('Alcohol Level by Quality')
plt.xlabel('0 = Low Quality, 1 = High Quality')
plt.show()

### Step 8 - Preprocess & Train Model 2

In [ ]:
X2 = df2.drop('target', axis=1)
y2 = df2['target']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train)
X2_test  = scaler2.transform(X2_test)

model2 = LogisticRegression(max_iter=10000)
model2.fit(X2_train, y2_train)

print('Model 2 trained!')

### Step 9 - Evaluate Model 2

In [ ]:
y2_pred = model2.predict(X2_test)
acc2 = accuracy_score(y2_test, y2_pred)

print('Accuracy:', round(acc2 * 100, 2), '%')
print()
print(classification_report(y2_test, y2_pred, target_names=['Low Quality', 'High Quality']))

In [ ]:
cm2 = confusion_matrix(y2_test, y2_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Low Quality', 'High Quality'],
            yticklabels=['Low Quality', 'High Quality'])
plt.title('Confusion Matrix - Wine Quality')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

---
## Gradio UI
> Run this cell. A UI will open in your browser and also inside the notebook.
> Default values are already filled in. Just click **Predict**.

In [ ]:
import gradio as gr
import numpy as np

feature_names_1 = list(df1.drop('target', axis=1).columns)
feature_names_2 = list(df2.drop('target', axis=1).columns)

means1 = scaler1.mean_
means2 = scaler2.mean_

# Prediction function for Breast Cancer
def predict_cancer(mean_radius, mean_texture, mean_perimeter, mean_area, mean_smoothness):
    # Use 5 inputs, fill remaining 25 features with average values
    full = list(means1)
    full[0] = mean_radius
    full[1] = mean_texture
    full[2] = mean_perimeter
    full[3] = mean_area
    full[4] = mean_smoothness
    
    data = np.array(full).reshape(1, -1)
    data = scaler1.transform(data)
    pred = model1.predict(data)[0]
    prob = model1.predict_proba(data)[0]
    
    if pred == 1:
        result = 'Benign (Non-Cancerous)'
    else:
        result = 'Malignant (Cancerous)'
    
    return f'Prediction: {result}\nMalignant: {prob[0]*100:.1f}%  |  Benign: {prob[1]*100:.1f}%'

# Prediction function for Wine Quality
def predict_wine(fixed_acidity, volatile_acidity, citric_acid, residual_sugar,
                 chlorides, free_so2, total_so2, density, pH, sulphates, alcohol):
    data = np.array([[fixed_acidity, volatile_acidity, citric_acid, residual_sugar,
                      chlorides, free_so2, total_so2, density, pH, sulphates, alcohol]])
    data = scaler2.transform(data)
    pred = model2.predict(data)[0]
    prob = model2.predict_proba(data)[0]
    
    if pred == 1:
        result = 'High Quality (score >= 7)'
    else:
        result = 'Low Quality (score < 7)'
    
    return f'Prediction: {result}\nLow: {prob[0]*100:.1f}%  |  High: {prob[1]*100:.1f}%'

# Build the interface
with gr.Blocks(title='ANN Lab Task 02') as app:
    
    gr.Markdown('# ANN Lab Task 02 - Logistic Regression')
    
    with gr.Tabs():
        
        # Tab 1 - Breast Cancer
        with gr.Tab('Dataset 1 - Breast Cancer'):
            gr.Markdown(f'### Accuracy: {acc1*100:.2f}% | Enter 5 main features')
            
            r1 = gr.Number(label='Mean Radius',     value=round(float(means1[0]), 2))
            r2 = gr.Number(label='Mean Texture',    value=round(float(means1[1]), 2))
            r3 = gr.Number(label='Mean Perimeter',  value=round(float(means1[2]), 2))
            r4 = gr.Number(label='Mean Area',       value=round(float(means1[3]), 2))
            r5 = gr.Number(label='Mean Smoothness', value=round(float(means1[4]), 4))
            
            btn1 = gr.Button('Predict', variant='primary')
            out1 = gr.Textbox(label='Result')
            
            btn1.click(fn=predict_cancer, inputs=[r1, r2, r3, r4, r5], outputs=out1)
        
        # Tab 2 - Wine Quality
        with gr.Tab('Dataset 2 - Wine Quality'):
            gr.Markdown(f'### Accuracy: {acc2*100:.2f}% | Enter all 11 features')
            
            w1  = gr.Number(label='Fixed Acidity',    value=round(float(means2[0]),  2))
            w2  = gr.Number(label='Volatile Acidity', value=round(float(means2[1]),  3))
            w3  = gr.Number(label='Citric Acid',      value=round(float(means2[2]),  3))
            w4  = gr.Number(label='Residual Sugar',   value=round(float(means2[3]),  2))
            w5  = gr.Number(label='Chlorides',        value=round(float(means2[4]),  4))
            w6  = gr.Number(label='Free SO2',         value=round(float(means2[5]),  2))
            w7  = gr.Number(label='Total SO2',        value=round(float(means2[6]),  2))
            w8  = gr.Number(label='Density',          value=round(float(means2[7]),  4))
            w9  = gr.Number(label='pH',               value=round(float(means2[8]),  2))
            w10 = gr.Number(label='Sulphates',        value=round(float(means2[9]),  3))
            w11 = gr.Number(label='Alcohol',          value=round(float(means2[10]), 2))
            
            btn2 = gr.Button('Predict', variant='primary')
            out2 = gr.Textbox(label='Result')
            
            btn2.click(fn=predict_wine,
                       inputs=[w1, w2, w3, w4, w5, w6, w7, w8, w9, w10, w11],
                       outputs=out2)

app.launch(inline=True)

---
## Flask UI
> **Instructions:**
> 1. Run **Cell A** to save the Flask app file
> 2. Run **Cell B** to start the server
> 3. Open **http://localhost:5000** in your browser
> 4. Run **Cell C** to stop the server when done

In [ ]:
# Cell A - Save models and write Flask app file
import pickle

pickle.dump(model1,  open('model1.pkl',  'wb'))
pickle.dump(scaler1, open('scaler1.pkl', 'wb'))
pickle.dump(model2,  open('model2.pkl',  'wb'))
pickle.dump(scaler2, open('scaler2.pkl', 'wb'))

flask_code = """
import pickle
import numpy as np
from flask import Flask, request, jsonify, render_template_string

app = Flask(__name__)

model1  = pickle.load(open('model1.pkl',  'rb'))
scaler1 = pickle.load(open('scaler1.pkl', 'rb'))
model2  = pickle.load(open('model2.pkl',  'rb'))
scaler2 = pickle.load(open('scaler2.pkl', 'rb'))

means1 = scaler1.mean_.tolist()
means2 = scaler2.mean_.tolist()

PAGE = open('flask_page.html').read()

@app.route('/')
def home():
    return render_template_string(PAGE)

@app.route('/predict_cancer', methods=['POST'])
def predict_cancer():
    d = request.get_json()
    full = list(means1)
    vals = d['values']
    for i in range(len(vals)):
        full[i] = float(vals[i])
    data = np.array(full).reshape(1, -1)
    data = scaler1.transform(data)
    pred = int(model1.predict(data)[0])
    prob = model1.predict_proba(data)[0].tolist()
    label = 'Benign (Non-Cancerous)' if pred == 1 else 'Malignant (Cancerous)'
    return jsonify({'label': label, 'prob0': round(prob[0]*100, 1), 'prob1': round(prob[1]*100, 1)})

@app.route('/predict_wine', methods=['POST'])
def predict_wine():
    d = request.get_json()
    vals = [float(x) for x in d['values']]
    data = np.array(vals).reshape(1, -1)
    data = scaler2.transform(data)
    pred = int(model2.predict(data)[0])
    prob = model2.predict_proba(data)[0].tolist()
    label = 'High Quality (score >= 7)' if pred == 1 else 'Low Quality (score < 7)'
    return jsonify({'label': label, 'prob0': round(prob[0]*100, 1), 'prob1': round(prob[1]*100, 1)})

if __name__ == '__main__':
    app.run(port=5000, debug=False, use_reloader=False)
"""

with open('flask_app.py', 'w') as f:
    f.write(flask_code)

print('Flask app saved as flask_app.py')
print('Models saved!')

In [ ]:
# Write the HTML page for Flask
html = """<!DOCTYPE html>
<html>
<head>
  <title>ANN Lab Task 02 - Logistic Regression</title>
  <style>
    body { font-family: Arial; background: #f4f4f4; padding: 20px; }
    h1   { color: #333; text-align: center; }
    .tab-btns { text-align: center; margin: 20px; }
    button.tab { padding: 10px 30px; margin: 5px; font-size: 16px;
                 border: none; cursor: pointer; border-radius: 6px; }
    #tab1-btn { background: #e74c3c; color: white; }
    #tab2-btn { background: #9b59b6; color: white; }
    .box { max-width: 600px; margin: auto; background: white;
           padding: 25px; border-radius: 10px;
           box-shadow: 0 2px 10px rgba(0,0,0,0.1); }
    .hidden { display: none; }
    label   { font-weight: bold; }
    input   { width: 100%; padding: 8px; margin: 6px 0 14px;
              border: 1px solid #ccc; border-radius: 5px; font-size: 14px; }
    .predict-btn { background: #2ecc71; color: white; border: none;
                   padding: 12px 30px; font-size: 16px;
                   border-radius: 6px; cursor: pointer; width: 100%; }
    .result { margin-top: 16px; padding: 14px; border-radius: 8px;
              background: #eaf7fb; border: 1px solid #aed6f1;
              font-size: 16px; font-weight: bold; text-align: center; }
  </style>
</head>
<body>

<h1>ANN Lab Task 02 - Logistic Regression</h1>

<div class="tab-btns">
  <button class="tab" id="tab1-btn" onclick="showTab(1)">Dataset 1 - Breast Cancer</button>
  <button class="tab" id="tab2-btn" onclick="showTab(2)">Dataset 2 - Wine Quality</button>
</div>

<!-- TAB 1 -->
<div class="box" id="tab1">
  <h2>Breast Cancer Prediction</h2>
  <p>Enter the 5 main features. Other features use average values.</p>
  <label>Mean Radius</label>
  <input type="number" id="c1" value="14.13" step="any">
  <label>Mean Texture</label>
  <input type="number" id="c2" value="19.29" step="any">
  <label>Mean Perimeter</label>
  <input type="number" id="c3" value="91.97" step="any">
  <label>Mean Area</label>
  <input type="number" id="c4" value="654.89" step="any">
  <label>Mean Smoothness</label>
  <input type="number" id="c5" value="0.0964" step="any">
  <button class="predict-btn" onclick="predictCancer()">Predict</button>
  <div class="result" id="res1" style="display:none"></div>
</div>

<!-- TAB 2 -->
<div class="box hidden" id="tab2">
  <h2>Wine Quality Prediction</h2>
  <p>Enter all 11 wine properties.</p>
  <label>Fixed Acidity</label>    <input type="number" id="w1"  value="8.32" step="any">
  <label>Volatile Acidity</label> <input type="number" id="w2"  value="0.528" step="any">
  <label>Citric Acid</label>      <input type="number" id="w3"  value="0.271" step="any">
  <label>Residual Sugar</label>   <input type="number" id="w4"  value="2.54" step="any">
  <label>Chlorides</label>        <input type="number" id="w5"  value="0.087" step="any">
  <label>Free SO2</label>         <input type="number" id="w6"  value="15.87" step="any">
  <label>Total SO2</label>        <input type="number" id="w7"  value="46.47" step="any">
  <label>Density</label>          <input type="number" id="w8"  value="0.9967" step="any">
  <label>pH</label>               <input type="number" id="w9"  value="3.311" step="any">
  <label>Sulphates</label>        <input type="number" id="w10" value="0.658" step="any">
  <label>Alcohol</label>          <input type="number" id="w11" value="10.42" step="any">
  <button class="predict-btn" onclick="predictWine()">Predict</button>
  <div class="result" id="res2" style="display:none"></div>
</div>

<script>
function showTab(n) {
  document.getElementById('tab1').classList.toggle('hidden', n !== 1);
  document.getElementById('tab2').classList.toggle('hidden', n !== 2);
}

async function predictCancer() {
  const vals = ['c1','c2','c3','c4','c5'].map(id => document.getElementById(id).value);
  const res  = await fetch('/predict_cancer', {
    method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({values: vals})
  });
  const d = await res.json();
  const box = document.getElementById('res1');
  box.style.display = 'block';
  box.innerHTML = d.label + '<br><small>Malignant: ' + d.prob0 + '%  |  Benign: ' + d.prob1 + '%</small>';
}

async function predictWine() {
  const vals = ['w1','w2','w3','w4','w5','w6','w7','w8','w9','w10','w11']
               .map(id => document.getElementById(id).value);
  const res  = await fetch('/predict_wine', {
    method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({values: vals})
  });
  const d = await res.json();
  const box = document.getElementById('res2');
  box.style.display = 'block';
  box.innerHTML = d.label + '<br><small>Low: ' + d.prob0 + '%  |  High: ' + d.prob1 + '%</small>';
}
</script>
</body>
</html>"""

with open('flask_page.html', 'w') as f:
    f.write(html)

print('flask_page.html saved!')

In [ ]:
# Cell B - Start Flask Server
import subprocess, sys, time, webbrowser

flask_process = subprocess.Popen([sys.executable, 'flask_app.py'])
time.sleep(2)

print('Flask server started!')
print('Open: http://localhost:5000')
webbrowser.open('http://localhost:5000')

In [ ]:
# Cell C - Stop Flask Server
flask_process.terminate()
print('Flask server stopped.')